In [4]:
"""
================================================================================
 FINAL 10-STAGE STATISTICAL PIPELINE — crack image, SALT & PEPPER (impulse) noise
================================================================================
 CORRECTED FLOW (identify -> remove -> normalize -> filter). The key fix over the
 earlier version: Stage 9 normalises the STAGE-8 DENOISED image, not the noisy
 input, so the pipeline chains correctly and the normalised panel is clean.
   Stage 8 (decision-based median)  ACTIVE   -> removes the salt & pepper
   Stage 9 (min-max, on DENOISED)   conditioning of the cleaned image
   Stage 10 (NL-means)              SKIPPED  -> wrong tool for impulse
 Genuine clean reference -> exact full-reference metrics.
================================================================================
"""
import os, string, warnings
warnings.filterwarnings("ignore", message="findfont")
import numpy as np
import cv2
from scipy import ndimage, stats
from skimage.util import img_as_float, img_as_ubyte, random_noise
from skimage.restoration import estimate_sigma
from skimage.measure import shannon_entropy
from skimage.metrics import mean_squared_error as MSE
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.metrics import structural_similarity as SSIM
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
 
CLEAN = "org_img.jpg"
SRC   = "sp.png"
OUT   = "C:/pipeline_out/"; os.makedirs(OUT, exist_ok=True)
ref = cv2.imread(CLEAN, cv2.IMREAD_GRAYSCALE)
if ref is None:
    raise FileNotFoundError("Clean reference not found at CLEAN=%r (cwd=%s)" % (CLEAN, os.getcwd()))
if not os.path.exists(SRC):
    cv2.imwrite(SRC, img_as_ubyte(random_noise(img_as_float(ref), mode="s&p", amount=0.05, rng=0)))
def w(n, a): cv2.imwrite(OUT + n, np.clip(a, 0, 255).astype(np.uint8))
def local_mean_var(x, ww=7):
    m = ndimage.uniform_filter(x, ww); v = np.clip(ndimage.uniform_filter(x*x, ww) - m*m, 0, None) * (ww*ww)/(ww*ww-1); return m, v
def despeckle(y, T=60):
    med = ndimage.median_filter(y, 3).astype(float); m = np.abs(y.astype(float) - med) > T
    return np.where(m, med, y.astype(float)).astype(np.uint8), m
 
# ══ STAGE 1 — LOAD ══
img = cv2.imread(SRC, cv2.IMREAD_GRAYSCALE); imgf = img.astype(np.float64); flat = imgf.ravel(); N = img.size
w("s01_input.png", img); print("STAGE 1  load    : shape=%s  [cv2.imread]" % (img.shape,))
# ══ STAGE 2 — RANGE ══
vmin, vmax = int(img.min()), int(img.max()); q1, med_v, q3 = np.percentile(flat, [25, 50, 75])
print("STAGE 2  range   : min=%d max=%d  [np.percentile]" % (vmin, vmax))
# ══ STAGE 3 — CENTRAL TENDENCY ══
mean, median = flat.mean(), np.median(flat); tmean = stats.trim_mean(flat, 0.1); mode = float(stats.mode(img.ravel()).mode)
print("STAGE 3  centre  : mean=%.1f median=%.1f mode=%.0f  [scipy.stats]" % (mean, median, mode))
# ══ STAGE 4 — SPREAD ══
std = flat.std(); iqr = stats.iqr(flat); mad = stats.median_abs_deviation(flat, scale="normal")
print("STAGE 4  spread  : std=%.1f iqr=%.1f MAD=%.1f  [np.std, scipy.stats]" % (std, iqr, mad))
# ══ STAGE 5 — OUTLIER / IMPULSE IDENTIFICATION ══
med3 = ndimage.median_filter(img, 3).astype(float); salt = (imgf - med3) > 60; pep = (med3 - imgf) > 60; imp_mask = salt | pep
lab, ncl = ndimage.label(imp_mask); sizes = (np.sort(ndimage.sum(np.ones_like(lab), lab, range(1, ncl+1)))[::-1] if ncl else np.array([0]))
med_sz = float(np.median(sizes)) if ncl else 0.0; imp_frac = imp_mask.sum()/N
print("STAGE 5  outlier : impulse specks=%d (%.2f%%) salt=%d pepper=%d median-cluster=%.0fpx  [ndimage.label]"
      % (imp_mask.sum(), 100*imp_frac, int(salt.sum()), int(pep.sum()), med_sz))
# ══ STAGE 6 — SHAPE + NOISE DISTRIBUTION ══
hist = cv2.calcHist([img], [0], None, [256], [0, 256]).ravel().astype(np.int64); centers = np.arange(256)+0.5
sk, ku = stats.skew(flat), stats.kurtosis(flat); ent = shannon_entropy(img)
res_noise = (imgf - med3).ravel(); res_kurt = stats.kurtosis(res_noise)
print("STAGE 6  shape   : SCENE skew=%.2f kurt=%.2f | NOISE(residual) kurtosis=%.1f  [calcHist+residual]" % (sk, ku, res_kurt))
# ══ STAGE 7 — NOISE MODEL IDENTIFICATION ══
sig255 = estimate_sigma(img_as_float(img), channel_axis=None) * 255
lmean, lvar = local_mean_var(imgf, 7)
gmag = ndimage.gaussian_gradient_magnitude(imgf, 1)
usable = (~imp_mask) & (gmag < np.percentile(gmag, 50))
mm, vv = lmean[usable], lvar[usable]; ed = np.linspace(mm.min(), mm.max(), 22); cx, cy = [], []
for i in range(len(ed)-1):
    s=(mm>=ed[i])&(mm<ed[i+1])
    if s.sum()>80: cx.append(.5*(ed[i]+ed[i+1])); cy.append(np.median(vv[s]))
cx, cy = np.array(cx), np.array(cy); a_mv = np.polyfit(cx, cy, 1)[0] if len(cx)>1 else 0.0
is_impulse = (imp_frac > 0.001) and (med_sz <= 2)
model = "Impulse (salt & pepper)" if is_impulse else ("Additive Gaussian" if sig255 > 2 else "Clean")
print("STAGE 7  noiseID : sigma_hat=%.2f residual-kurt=%.1f mean-var-slope=%.3f => MODEL: %s" % (sig255, res_kurt, a_mv, model))
# ══ STAGE 8 — OUTLIER REMOVAL [decision-median] ACTIVE  (this is the CLEANED image) ══
if is_impulse:
    denoised, _ = despeckle(img)
else:
    denoised = img.copy()
w("s08_denoised.png", denoised)
cv2.imwrite(OUT + "cleaned_result.png", denoised)              # explicit cleaned output
print("STAGE 8  outrem  : ACTIVE decision-median -> %.2f%% px modified  [np.where+median]" % (100*imp_mask.mean()))
# ══ STAGE 9 — NORMALISE (affine) — OPERATES ON THE DENOISED IMAGE (corrected) ══
denf = denoised.astype(np.float64)
norm01 = (denf - denf.min()) / (denf.max() - denf.min())       # <-- fix: normalise the CLEANED image
w("s09_normalised.png", norm01 * 255)
print("STAGE 9  normal  : min-max [0,1] on the DENOISED image (affine)")
# ══ STAGE 10 — FILTER (NL-means) SKIPPED ══
print("STAGE 10 filter  : SKIPPED for impulse (median already removed the specks)")
 
# ══ METRICS (full-reference vs clean) ══
def Mt(x): x=np.clip(x,0,255).astype(np.uint8); return MSE(ref,x), PSNR(ref,x,data_range=255), SSIM(ref,x,data_range=255)
band = ndimage.binary_dilation(ref < (ref.mean()-ref.std()), iterations=3)
def bS(x): x=np.clip(x,0,255).astype(np.uint8); _,sm=SSIM(ref,x,data_range=255,full=True); return float(sm[band].mean())
mn,pn,sn=Mt(img); md,pd,sd=Mt(denoised); mse_np=float(np.mean((ref.astype(float)-denoised.astype(float))**2)); cn,cc=bS(img),bS(denoised)
# validation: specks before/after
_, m_after = despeckle(denoised); specks_after = int(m_after.sum())
K=10; P,S,C=[],[],[]
for k in range(K):
    ny=img_as_ubyte(random_noise(img_as_float(ref),mode="s&p",amount=0.05,rng=k)); dn,_=despeckle(ny); _,p,ss=Mt(dn); P.append(p);S.append(ss);C.append(bS(dn))
print("\nMETRICS (full-reference vs clean):")
print("  noisy    MSE=%.2f PSNR=%.2f SSIM=%.4f crack-SSIM=%.4f" % (mn,pn,sn,cn))
print("  denoised MSE=%.2f PSNR=%.2f SSIM=%.4f crack-SSIM=%.4f" % (md,pd,sd,cc))
print("  gain dPSNR=%+.2f dSSIM=%+.4f | specks %d->%d removed | expected %d draws PSNR %.2f±%.2f"
      % (pd-pn, sd-sn, imp_mask.sum(), specks_after, K, np.mean(P), np.std(P)))
 
# ══ FLOW FIGURE ══
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],"font.size":10,"mathtext.fontset":"stix"})
L=list(string.ascii_lowercase); fig,ax=plt.subplots(3,6,figsize=(30,15.5)); ax=ax.ravel()
def IMG(i,im,t,cmap="gray",vmn=0,vmx=255): ax[i].imshow(im,cmap=cmap,vmin=vmn,vmax=vmx); ax[i].set_title("(%s) %s"%(L[i],t),fontsize=10.5,fontweight="bold"); ax[i].set_xticks([]); ax[i].set_yticks([])
def GT(i,t): ax[i].set_title("(%s) %s"%(L[i],t),fontsize=10.5,fontweight="bold"); ax[i].grid(True,alpha=.3,lw=.6)
def bl(a,bars,fmt="%.1f"):
    for b in bars: a.annotate(fmt%b.get_height(),(b.get_x()+b.get_width()/2,b.get_height()),textcoords="offset points",xytext=(0,2),ha="center",va="bottom",fontsize=8.5,fontweight="bold")
IMG(0,img,"S1 · Input (salt & pepper)"); ax[1].bar(centers,hist,width=1,color="lightsteelblue"); GT(1,"S1 · Intensity histogram = SCENE"); ax[1].set_xlabel("Intensity [0–255]"); ax[1].set_ylabel("Count")
IMG(2,img,"S2 · Range")
bpx=ax[3].boxplot(flat,vert=True,whis=(0.1,99.9),widths=.5,patch_artist=True); bpx["boxes"][0].set_facecolor("lightsteelblue")
for yv,lbl in [(vmin,"min %d"%vmin),(med_v,"med %.0f"%med_v),(vmax,"max %d"%vmax)]: ax[3].annotate(lbl,(1.3,yv),fontsize=8.5,va="center",fontweight="bold")
GT(3,"S2 · Box plot"); ax[3].set_ylabel("Intensity [0–255]"); ax[3].set_xticks([])
IMG(4,img,"S3 · Central tendency"); b3=ax[5].bar(["mean","median","trim","mode"],[mean,median,tmean,mode],color=["crimson","navy","teal","green"]); bl(ax[5],b3); GT(5,"S3 · Estimators"); ax[5].set_ylabel("Intensity")
IMG(6,img,"S4 · Spread"); b4=ax[7].bar(["std","IQR","MAD"],[std,iqr,mad],color=["crimson","grey","green"]); bl(ax[7],b4); GT(7,"S4 · Classical vs robust"); ax[7].set_ylabel("Intensity units")
IMG(8,imp_mask,"S5 · Impulse mask (salt&pepper)",vmn=0,vmx=1); top=sizes[:10]; b5=ax[9].bar(range(1,len(top)+1),top,color="indianred"); bl(ax[9],b5,"%d"); GT(9,"S5 · Cluster sizes (all=1px)"); ax[9].set_xlabel("rank"); ax[9].set_ylabel("size [px]")
IMG(10,img,"S6 · Shape"); ax[11].hist(res_noise,bins=201,range=(-120,120),color="seagreen",log=True); GT(11,"S6 · NOISE dist (img−median) kurt=%.0f => impulse"%res_kurt); ax[11].set_xlabel("deviation from median"); ax[11].set_ylabel("Count (log)")
IMG(12,lvar,"S7 · Local-variance map",cmap="magma",vmn=float(lvar.min()),vmx=float(np.percentile(lvar,99)))
if len(cx)>1:
    ax[13].plot(cx,cy,"o",color="steelblue"); xs=np.linspace(cx.min(),cx.max(),30); ax[13].plot(xs,a_mv*xs+np.polyfit(cx,cy,1)[1],"crimson",lw=2,label="slope=%.3f"%a_mv); ax[13].legend(fontsize=8)
GT(13,"S7 · mean-var + residual-kurt %.0f => %s"%(res_kurt,model.split()[0])); ax[13].set_xlabel("local mean"); ax[13].set_ylabel("local variance")
IMG(14,denoised,"S8 · Denoised (decision-median, ACTIVE)"); b8=ax[15].bar(["noisy","denoised"],[mn,md],color=["indianred","seagreen"]); bl(ax[15],b8,"%.0f"); GT(15,"S8 · MSE vs clean"); ax[15].set_ylabel("MSE")
IMG(16,norm01,"S9 · Normalised DENOISED image [0,1]",vmn=0,vmx=1); xin=np.arange(256); yout=np.clip((xin-denf.min())/(denf.max()-denf.min()),0,1); ax[17].plot(xin,yout,color="darkorange",lw=2.2); GT(17,"S9 · Transfer function (affine)"); ax[17].set_xlabel("input [0–255]"); ax[17].set_ylabel("output [0,1]")
for a in ax: a.add_patch(Rectangle((0,0),1,1,transform=a.transAxes,fill=False,edgecolor="#444",lw=1.5,clip_on=False,zorder=20))
plt.tight_layout(rect=[0.006,0.022,0.994,0.972]); fig.add_artist(Rectangle((0.006,0.02),0.988,0.958,transform=fig.transFigure,fill=False,edgecolor="black",lw=3,zorder=1000))
plt.savefig(OUT+"crack_sp_flow_corrected.png", dpi=100, bbox_inches="tight"); plt.close()
print("\nSaved corrected flow figure + cleaned_result.png to", os.path.abspath(OUT))

STAGE 1  load    : shape=(227, 227)  [cv2.imread]
STAGE 2  range   : min=0 max=255  [np.percentile]
STAGE 3  centre  : mean=164.5 median=171.0 mode=175  [scipy.stats]
STAGE 4  spread  : std=35.4 iqr=25.0 MAD=17.8  [np.std, scipy.stats]
STAGE 5  outlier : impulse specks=2555 (4.96%) salt=1282 pepper=1273 median-cluster=1px  [ndimage.label]
STAGE 6  shape   : SCENE skew=-2.24 kurt=10.18 | NOISE(residual) kurtosis=23.8  [calcHist+residual]
STAGE 7  noiseID : sigma_hat=4.98 residual-kurt=23.8 mean-var-slope=-5.851 => MODEL: Impulse (salt & pepper)
STAGE 8  outrem  : ACTIVE decision-median -> 4.96% px modified  [np.where+median]
STAGE 9  normal  : min-max [0,1] on the DENOISED image (affine)
STAGE 10 filter  : SKIPPED for impulse (median already removed the specks)

METRICS (full-reference vs clean):
  noisy    MSE=899.92 PSNR=18.59 SSIM=0.2634 crack-SSIM=0.3522
  denoised MSE=1.85 PSNR=45.45 SSIM=0.9937 crack-SSIM=0.9972
  gain dPSNR=+26.87 dSSIM=+0.7303 | specks 2555->0 removed | expected